In [16]:
import pandas as pd 
df = pd.read_parquet('arxiv_with_citations_dataset_full.parquet.gzip')
df

,arxiv_id,title,abstract,categories_list,year,article_url,repo_url_list,authors_list,citations,citation_source
0,0705.4676,Recursive n-gram hashing is pairwise independe...,Many applications use sequences of n consecu...,"[cs.DB, cs.CL]",2016,https://arxiv.org/pdf/0705.4676.pdf,"[https://github.com/zhaoxiaofei/bindash, https...","[Lemire Daniel, Kaser Owen]",26,semantic_scholar
1,0804.4451,Dependence Structure Estimation via Copula,Dependence strucuture estimation is one of t...,"[cs.LG, cs.IR, stat.ME]",2019,https://arxiv.org/pdf/0804.4451.pdf,[https://github.com/majianthu/dse],"[Ma Jian, Sun Zengqi]",9,semantic_scholar
2,0811.3301,Faster Retrieval with a Two-Pass Dynamic-Time-...,The Dynamic Time Warping (DTW) is a popular ...,"[cs.DB, cs.CV]",2012,https://arxiv.org/pdf/0811.3301.pdf,[https://github.com/lemire/lbimproved],[Lemire Daniel],193,semantic_scholar
3,0902.4682,Lectures on Jacques Herbrand as a Logician,We give some lectures on the work on formal ...,"[cs.LO, cs.AI]",2014,https://arxiv.org/pdf/0902.4682.pdf,[https://github.com/thejohncrafter/flows],"[Wirth Claus-Peter, Siekmann Joerg, Benzmuelle...",4,semantic_scholar
4,0906.2027,Matrix Completion from Noisy Entries,"Given a matrix M of low-rank, we consider th...","[cs.LG, stat.ML]",2012,https://arxiv.org/pdf/0906.2027.pdf,[https://github.com/jasonsun0310/MatrixComplet...,"[Keshavan Raghunandan H., Montanari Andrea, Oh...",726,semantic_scholar
...,...,...,...,...,...,...,...,...,...,...
123115,2507.15351,One Step is Enough: Multi-Agent Reinforcement ...,On-demand ride-sharing platforms face the fund...,"[cs.AI, cs.ET, cs.MA]",2025,https://arxiv.org/pdf/2507.15351.pdf,[https://github.com/RS2002/OSPO],"[Zhao Zijian, Li Sen]",1,semantic_scholar
123116,2507.15454,ObjectGS: Object-aware Scene Reconstruction an...,3D Gaussian Splatting is renowned for its high...,"[cs.GR, cs.AI, cs.CV, cs.HC]",2025,https://arxiv.org/pdf/2507.15454.pdf,[https://github.com/RuijieZhu94/ObjectGS],"[Zhu Ruijie, Yu Mulin, Xu Linning, Jiang Lihan...",2,semantic_scholar
123117,2507.15641,Leveraging Context for Multimodal Fallacy Clas...,"In this paper, we present our submission to th...","[cs.CL, cs.AI]",2025,https://arxiv.org/pdf/2507.15641.pdf,[https://github.com/alessiopittiglio/mm-argfal...,[Pittiglio Alessio],0,semantic_scholar
123118,cs/0212008,Principal Manifolds and Nonlinear Dimension Re...,Nonlinear manifold learning from unorganized...,"[cs.LG, cs.AI]",2016,https://arxiv.org/pdf/cs/0212008.pdf,[https://github.com/gitr00ki3/vpw],"[Zhang Zhenyue, Zha Hongyuan]",52,semantic_scholar


In [17]:
most_cited = df[df["citations"] >= 200].copy()
most_cited.reset_index(drop=True, inplace=True)
most_cited.to_parquet('arxiv_most_cited_dataset.parquet.gzip', compression='gzip')
len(most_cited)

10457

### Split the best 10k articles into tiers by citation count
This will prove helpful when upserting records to database. We will start with the most important tier
and scale up in amount of aticles along the way.

In [23]:
tier_I = most_cited[most_cited["citations"] >= 10000].copy() # ~ 100 papers
tier_II = most_cited[(most_cited["citations"] < 10000) & (most_cited["citations"] >= 1600)].copy() # ~ 1k papers
tier_III = most_cited[(most_cited["citations"] < 1600) & (most_cited["citations"] >= 500)].copy() # ~ 3k papers
tier_IV = most_cited[most_cited["citations"] < 500].copy() # ~ 6k papers
tier_I.reset_index(drop=True, inplace=True)
tier_II.reset_index(drop=True, inplace=True)
tier_III.reset_index(drop=True, inplace=True)
tier_IV.reset_index(drop=True, inplace=True)
print(len(tier_I))
print(len(tier_II))
print(len(tier_III))
print(len(tier_IV))
print(f"Length total: {len(tier_I) + len(tier_II) + len(tier_III) + len(tier_IV)}, expected: {len(most_cited)}")

114
965
3137
6241
Length total: 10457, expected: 10457


# Save the tier datasets to parquet

In [24]:
tier_I.to_parquet('arxiv_most_cited_tier_I.parquet.gzip', compression='gzip') # ~ 100 papers
tier_II.to_parquet('arxiv_most_cited_tier_II.parquet.gzip', compression='gzip') # ~ 1k papers
tier_III.to_parquet('arxiv_most_cited_tier_III.parquet.gzip', compression='gzip') # ~ 3k papers
tier_IV.to_parquet('arxiv_most_cited_tier_IV.parquet.gzip', compression='gzip') # ~ 6k papers